# Day 13(M3 Day04) 실습 2 — 질의 개선

**목표**: 사용자의 질문을 그대로 검색하지 않고 LLM으로 다듬어 검색 품질을 높인다. 실습 1의 한계 ①(표현에 따라 검색이 흔들림)을 다룬다.

| Part | 기법 | 한 줄 요약 |
| --- | --- | --- |
| Part 1 | 질의 재작성 (Query Rewriting) | 구어체 질문을 보고서 용어로 다시 쓴다 |
| Part 2 | HyDE | 답변처럼 생긴 가상 문서를 먼저 쓰고, 그 문서로 검색한다 |
| Part 3 | Multi-Query | 질문 하나를 여러 관점의 질문으로 늘려 각각 검색한다 |
| Part 4 | RAG Fusion | 여러 검색 결과를 RRF로 합쳐 순위를 다시 매긴다 |
| Part 5 | 체인으로 묶기 | 재작성·Multi-Query를 RAG 체인에 넣는다 |

**선수**: 실습 1(`Day13-1-NaiveRAG한계와문서전처리-실습.ipynb`)을 끝까지 실행해 `ai_trend_md` 저장소를 만들어 둔다.

**데이터**: 한국지능정보사회진흥원(NIA) AI@Data Report 2025-제3호 「토픽 분석을 통한 AI 주요 트렌드 및 2026 전망」(2025.12, 22쪽). 과학기술정보통신부·NIA '초거대 AI 데이터 발굴 및 기획' 사업의 산출물이다. 보고서 규정에 따라 무단전재를 금하고, 가공·인용할 때는 출처를 「한국지능정보사회진흥원(NIA)」으로 밝힌다. 저장소에는 넣지 않고 Day13-1이 NIA 공식 주소에서 `data/ai_trend_2026.pdf`로 내려받는다.

## 0. 환경 준비

In [1]:
import os, re
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

load_dotenv()
CONN = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

# PGVector마다 연결 풀을 새로 만들지 않도록 엔진 하나를 같이 쓴다 (Session pooler는 연결 15개 한도)
engine = create_engine(CONN, pool_size=2, max_overflow=0)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # temperature=0: 결과 재현성

def open_store(name):
    """Day13-1에서 만든 컬렉션을 연다. 비어 있으면 Day13-1을 먼저 실행하라고 알린다."""
    store = PGVector(embeddings=embeddings, collection_name=name, connection=engine, use_jsonb=True)
    if not store.similarity_search("AI", k=1):
        raise RuntimeError(f"'{name}' 컬렉션이 비어 있다. Day13-1 노트북을 먼저 끝까지 실행한다.")
    return store

# 출력 형식
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


# 검색결과 비교함수
def show(docs, n=120):
    for i, d in enumerate(docs, 1):
        m = d.metadata
        print(f"  [{i}] {m.get('page', '?')}쪽 #{m.get('chunk_id', '?')} | {d.page_content[:n].replace(chr(10), ' ')}")


print("ai_trend_md 저장소 준비 완료")

ai_trend_md 저장소 준비 완료


In [2]:
md_store = open_store("ai_trend_md")
retriever = md_store.as_retriever(search_kwargs={"k":3})

## Part 1. 질의 재작성 (Query Rewriting)

실습 1의 1-3에서 "내년 AI 모델은…"처럼 보고서에 없는 표현을 쓰면 검색이 흔들렸다. 질의 재작성은 사용자의 구어체·모호한 질문을 **문서에 실제로 쓰인 용어**로 다시 쓴다. 보고서의 핵심 용어를 프롬프트에 알려 준다.

In [3]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 검색 질의를 다듬는 전문가다.
    사용자의 구어체, 모호한 질문은 보고서 검색에 사용하도록 명확한 질의로 다시 쓴다.
    보고서 용어를 알맞게 사용한다.
    보고서 용어: AI 산업, AI 기술, AI 정책, 2026년 전망, 토픽모델링 등
    다시 작성한 질의만 출력한다.
"""),
    ("human", "원본 질문: {question}"),
])

# 질의재작성 프롬프트
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

for q in ["내년 AI는 어떻게 바뀌어요?", "AI가 회사 일에 어떻게 들어와요?", "AI 규제는 왜 중요해졌나요?"]:
    print(f"원본 : {q} \n 재작성 : {rewrite_chain.invoke({'question': q})}\n")

원본 : 내년 AI는 어떻게 바뀌어요? 
 재작성 : 2024년 AI 산업의 변화 전망은 무엇인가요?

원본 : AI가 회사 일에 어떻게 들어와요? 
 재작성 : AI 기술이 기업 운영에 어떻게 적용되고 있는지에 대한 사례와 분석.

원본 : AI 규제는 왜 중요해졌나요? 
 재작성 : AI 규제의 중요성에 대한 분석은 무엇인가요?



### 1-2. 오류 대응 — 재작성이 질문의 뜻을 바꾸면

출력에서 "내년"이 몇 년으로 바뀌었는지 본다. 모델은 이 보고서가 언제 나왔는지 모른다. 그래서 "내년"을 자기가 아는 시점으로 해석한다. 이 보고서는 2025년 12월에 나왔고 2026년을 전망한다. 기준 시점을 프롬프트에 알려 준다.

In [16]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 검색 질의를 다듬는 전문가다.
사용자의 구어체·모호한 질문을 보고서 검색에 알맞은 명확한 질의로 다시 쓴다.
이 보고서는 2025년 12월에 발행되었다. '올해'는 2025년, '내년'은 2026년으로 바꾼다.
보고서 용어를 알맞게 쓴다: AI 산업, AI 기술, AI 정책, 2026년 전망, 토픽모델링, 핵심 이슈,
AI 에이전트, 멀티모달, 추론형 AI, 온디바이스 AI, 안전성, 책임성, 규제 거버넌스, AI 기본법.
질문의 뜻을 바꾸거나 질문에 없는 조건을 더하지 않는다. 다시 쓴 질의만 출력한다.
"""),
    ("human", "원본 질문: {question}"),
])
# TODO: rewrite_prompt, llm, StrOutputParser를 이어 재작성 체인을 만드세요
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

for q in ["내년 AI는 어떻게 바뀌어요?", "AI가 회사 일에 어떻게 들어와요?", "AI 규제는 왜 중요해졌나요?"]:
    print(f"원본 : {q} \n 재작성 : {rewrite_chain.invoke({'question': q})}\n")

원본 : 내년 AI는 어떻게 바뀌어요? 
 재작성 : 2026년 AI 산업은 어떻게 변화할 것으로 예상되나요?

원본 : AI가 회사 일에 어떻게 들어와요? 
 재작성 : AI가 기업 업무에 어떻게 적용되고 있나요?

원본 : AI 규제는 왜 중요해졌나요? 
 재작성 : AI 규제의 중요성은 무엇인가요?



> **주의:** 질의 재작성은 질문을 **다른 문장으로 바꾸는** 일이라 뜻이 바뀔 위험이 늘 있다. 날짜처럼 상대적인 표현("내년", "최근"), 문서가 가정하는 배경을 프롬프트에 알려 주고, 재작성 결과를 출력해 확인한다.

### 1-3. 재작성 전후 검색 비교

In [ ]:
original = "내년 AI 모델은 어떻게 발전해?"




보고서 18쪽 "2026년 AI 기술 전망"이 답이 있는 곳이다. 이번 실행에서 두 검색 모두 18쪽이 1위였다. 차이는 2·3위다.

| 검색어 | 2위 | 3위 |
| --- | --- | --- |
| 원본 ("내년 AI 모델은…") | 20쪽 (데이터 생태계 대응) | 2쪽 (보고서 시리즈 소개) |
| 재작성 ("2026년 AI 모델의…") | 17쪽 (2026년 전망 장) | 5쪽 (기술 트렌드 요약) |

원본 질문은 답과 상관없는 보고서 소개 쪽까지 가져왔고, 재작성 질의는 전망을 다룬 쪽을 모았다.

## Part 2. HyDE — 가상 답변으로 검색하기

HyDE(Hypothetical Document Embeddings)란: 질문으로 바로 검색하지 않고, LLM이 **답변처럼 생긴 가상 문서**를 먼저 쓰게 한 뒤 그 문서로 검색하는 방법이다. 질문은 짧고 문서는 설명문이라 둘의 모양이 다르다. 가상 문서는 문서와 모양이 비슷해서 더 가까운 조각을 찾는다.

| 방법 | 검색에 쓰는 문장 | 모양 |
| --- | --- | --- |
| 원본 질문 | 사용자 질문 | 짧은 질문 |
| 질의 재작성 | 다듬은 질문 | 질문 |
| HyDE | 가상 답변 문단 | 보고서 같은 설명문 |

> **주의:** 가상 문서의 내용은 사실이 아닐 수 있다. **검색에만** 쓰고, 답변 근거로는 실제로 검색된 조각만 쓴다.

In [ ]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 분석가다.
질문에 대해 보고서에 들어 있을 법한 설명문을 3~4문장으로 쓴다.
AI 산업·기술·정책, 2026년 전망, 토픽모델링, 핵심 이슈 같은 보고서 용어를 쓴다.
답을 확정하지 말고, 검색에 쓸 가상 문서처럼 쓴다."""),
    ("human", "{question}"),
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

question = "내년 AI 기술은 어떤 방향으로 발전할까요?"
hypothetical = hyde_chain.invoke({"question": question})
print("[가상 문서]\n" + hypothetical)

print("\n[원본 질문으로 검색]")
show(retriever.invoke(question))
print("\n[가상 문서로 검색]")
# TODO: 가상 문서(hypothetical)로 검색해 show로 출력하세요

## Part 3. Multi-Query — 질문을 여러 관점으로 늘리기

질문 하나로는 한 방향의 조각만 찾는다. Multi-Query는 LLM이 질문을 **여러 관점의 질문**으로 늘리고, 각 질문으로 검색한 결과를 합친다.

### 3-1. LangChain 내장 MultiQueryRetriever

`MultiQueryRetriever`는 질문을 여러 개로 바꿔 각각 검색하고, 겹치는 조각은 한 번만 남긴다.

In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever

multi_retriever = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)

question = "AI가 회사 일에 어떻게 들어와요?"
print("[원본 질문 하나로]", len(retriever.invoke(question)), "개")
results = multi_retriever.invoke(question)
print("[Multi-Query]", len(results), "개 (중복 제거 후)")
show(results)

### 3-2. 직접 만들기 — 산업·기술·정책 세 관점

내장 기능은 질문을 어떻게 늘렸는지 보이지 않는다. 보고서 구성(산업·기술·정책)에 맞춰 직접 늘리고, 겹치는 조각을 지운다.

In [ ]:
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 AI 트렌드 보고서 검색 전문가다.
질문을 아래 세 관점의 검색 질의로 바꾼다. 한 줄에 하나씩, 번호나 설명 없이 질의만 쓴다.
- 산업 관점: AI 도입, 시장 규모, 기업 운영, AI 에이전트
- 기술 관점: 멀티모달, 추론형 AI, 온디바이스 AI, 모델 고도화
- 정책 관점: AI 규제, 안전성, 책임성, 거버넌스, AI 기본법"""),
    ("human", "{question}"),
])
query_gen_chain = multi_query_prompt | llm | StrOutputParser()

def generate_queries(question):
    lines = query_gen_chain.invoke({"question": question}).split("\n")
    return [question] + [l.strip() for l in lines if l.strip()]      # 원본 질문도 포함한다

def multi_query_retrieve(question):
    seen, docs = set(), []
    for q in generate_queries(question):
        for d in retriever.invoke(q):
            # TODO: 처음 보는 조각(page_content 기준)만 seen에 기록하고 docs에 추가하세요
            pass
    return docs

question = "AI가 회사 일에 어떻게 들어와요?"
for i, q in enumerate(generate_queries(question)):
    print(f"  질의 {i}: {q}")
docs = multi_query_retrieve(question)
print(f"\n검색 조각 {len(docs)}개 (중복 제거 후)")
show(docs)

## Part 4. RAG Fusion — 여러 검색 결과를 RRF로 합치기

3-2는 조각을 **나온 순서대로** 합쳤다. 여러 질의에서 공통으로 높은 순위에 나온 조각이 더 중요할 가능성이 크다. RAG Fusion은 여러 검색 결과를 RRF(Reciprocal Rank Fusion)로 합쳐 순위를 다시 매긴다. 실습 3의 하이브리드 검색(`EnsembleRetriever`)이 BM25와 의미 검색을 합칠 때 쓰는 것과 같은 방법이다.

```text
RRF 점수 = Σ 1 / (60 + 순위)      (조각이 나온 검색 결과마다 더한다)
```

In [ ]:
def reciprocal_rank_fusion(results_list, rrf_k=60):
    scores, doc_map = {}, {}
    for results in results_list:
        for rank, d in enumerate(results, start=1):
            # TODO: 조각의 RRF 점수에 1 / (rrf_k + rank)를 더하세요 (처음 나온 조각은 0에서 시작)
            doc_map[d.page_content] = d
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [(doc_map[k], scores[k]) for k in ranked]

question = "AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘."
queries = generate_queries(question)
fused = reciprocal_rank_fusion([retriever.invoke(q) for q in queries])

print(f"질의 {len(queries)}개 → 합친 조각 {len(fused)}개\n")
for d, score in fused[:5]:
    print(f"  RRF {score:.4f} | {d.metadata['page']}쪽 #{d.metadata['chunk_id']} | {d.page_content[:60].replace(chr(10), ' ')}")

점수가 높은 조각은 여러 질의에서 **모두** 앞쪽에 나온 조각이다. 한 질의에서만 1위를 한 조각보다 앞설 수 있다.

## Part 5. 체인으로 묶기

재작성과 Multi-Query를 RAG 체인에 넣는다. 검색은 다듬은 질문으로 하고, 답변은 **원래 질문**에 맞춰 한다. `RunnablePassthrough.assign`은 들어온 딕셔너리에 키를 하나씩 더한다(Day12 확장 W-8과 같은 방식).

In [ ]:
from operator import itemgetter

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서에 근거해서만 답한다. 문서에서 확인할 수 없으면 '문서에서 확인할 수 없습니다'라고 답한다.\n\n[문서]\n{context}"),
    ("human", "{question}"),
])

rewrite_rag_chain = None

multi_rag_chain = None

q = "AI가 회사 일에 어떻게 들어와요?"




### 관찰

| 기법 | 한 질문에 드는 LLM 호출 | 좋아지는 경우 | 조심할 점 |
| --- | --- | --- | --- |
| 질의 재작성 | 답변 + 재작성 1번 | 구어체·모호한 질문 | 재작성이 질문 뜻을 바꿀 수 있다 |
| HyDE | 답변 + 가상 문서 1번 | 짧은 질문과 긴 설명문 문서 | 가상 문서의 내용은 검색에만 쓴다 |
| Multi-Query | 답변 + 질의 생성 1번 (검색은 질의 수만큼) | 여러 측면을 묻는 질문 | 조각이 늘어나 토큰이 많아진다 |
| RAG Fusion | Multi-Query와 같음 | 여러 질의에서 공통으로 중요한 조각 찾기 | 순위 계산이 한 번 더 든다 |

> **참고:** 어느 기법이 이 문서와 질문에서 실제로 나은지는 출력만 보고 판단하기 어렵다. 실습 4에서 평가 세트로 재 본다.

## 확인 문제

1. 질의 재작성과 HyDE는 검색에 쓰는 문장이 각각 무엇인가?
2. HyDE의 가상 문서를 답변 근거로 쓰면 안 되는 이유는?
3. Multi-Query에서 원본 질문도 검색에 포함한 이유는?
4. RRF는 무엇을 기준으로 조각의 순위를 다시 매기는가?
5. 재작성 RAG 체인에서 검색은 다듬은 질문으로 하고 답변은 원래 질문으로 하는 이유는?